In [3]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

from dotenv import load_dotenv
load_dotenv()

True

In [4]:
# 12 documents spanning health, programming, history, and nature
# Docs 1-2: contain the exact word "vaccine" — BM25 keyword match
# Docs 3-5: semantically related (immune system, antibodies, herd immunity) but lack the word "vaccine"
#            — dense search finds these, BM25 misses them
# Docs 6-12: off-topic
docs = [
    Document(page_content="Vaccines work by introducing a weakened or inactivated pathogen to trigger an immune response.", metadata={"topic": "health"}),
    Document(page_content="The flu vaccine is reformulated each year to match the most prevalent circulating virus strains.", metadata={"topic": "health"}),
    Document(page_content="The immune system produces antibodies that recognise and neutralise foreign pathogens in the body.", metadata={"topic": "health"}),
    Document(page_content="Herd immunity occurs when enough of a population becomes resistant to a disease, slowing its spread.", metadata={"topic": "health"}),
    Document(page_content="White blood cells called B-lymphocytes produce proteins that bind to and destroy specific antigens.", metadata={"topic": "health"}),
    Document(page_content="Version control systems like Git track changes to code and enable collaboration across teams.", metadata={"topic": "programming"}),
    Document(page_content="Docker containers package applications with their dependencies for consistent deployment.", metadata={"topic": "programming"}),
    Document(page_content="The French Revolution began in 1789 and fundamentally transformed European political structures.", metadata={"topic": "history"}),
    Document(page_content="The Silk Road was an ancient trade network connecting China to the Mediterranean world.", metadata={"topic": "history"}),
    Document(page_content="The Amazon rainforest produces about 20% of the world's oxygen and houses 10% of all species.", metadata={"topic": "nature"}),
    Document(page_content="Coral reefs cover less than 1% of the ocean floor but support about 25% of all marine species.", metadata={"topic": "nature"}),
    Document(page_content="REST APIs communicate over HTTP using standard methods like GET, POST, PUT, and DELETE.", metadata={"topic": "programming"}),
]

In [5]:
# Dense retriever: ChromaDB with OpenAI embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [6]:
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding = embeddings,
    collection_name='hybrid_search'
)

chroma_retriever = vectorstore.as_retriever(
    search_type = 'similarity',
    search_kwargs = {'k': 4}
)

bm25_retriever = BM25Retriever.from_documents(
    documents=docs,
    k=2,
    bm25_variant='plus'
)

In [7]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[chroma_retriever, bm25_retriever],
    weights=[0.8,0.2]
)

In [ ]:
query = "How do vaccines work to protect against diseases?"

bm25_results = bm25_retriever.invoke(query)
chroma_results = chroma_retriever.invoke(query)
ensemble_results = ensemble_retriever.invoke(query)

print("=== BM25 Only (keyword match) ===")
for i, doc in enumerate(bm25_results, 1):
    print(f"  [{i}] topic={doc.metadata['topic']}: {doc.page_content}")
print()

print("=== ChromaDB Only (semantic match) ===")
for i, doc in enumerate(chroma_results, 1):
    print(f"  [{i}] topic={doc.metadata['topic']}: {doc.page_content}")
print()

# Ensemble combines both — surfaces keyword matches AND semantic matches
print("=== Ensemble / Hybrid (keyword + semantic) ===")
for i, doc in enumerate(ensemble_results, 1):
    print(f"  [{i}] topic={doc.metadata['topic']}: {doc.page_content}")

=== BM25 Only (keyword match) ===
  [1] topic=health: Vaccines work by introducing a weakened or inactivated pathogen to trigger an immune response.
  [2] topic=programming: REST APIs communicate over HTTP using standard methods like GET, POST, PUT, and DELETE.

=== ChromaDB Only (semantic match) ===
  [1] topic=health: Vaccines work by introducing a weakened or inactivated pathogen to trigger an immune response.
  [2] topic=health: The immune system produces antibodies that recognise and neutralise foreign pathogens in the body.
  [3] topic=health: The flu vaccine is reformulated each year to match the most prevalent circulating virus strains.
  [4] topic=health: White blood cells called B-lymphocytes produce proteins that bind to and destroy specific antigens.

=== Ensemble / Hybrid (keyword + semantic) ===
  [1] topic=health: Vaccines work by introducing a weakened or inactivated pathogen to trigger an immune response.
  [2] topic=health: The immune system produces antibodies that r